<a href="https://colab.research.google.com/github/emmanuelyardamah/Data-Warehouse-and-Data-Mining/blob/main/Cleaning%20Dataset%20Process%20CAT%201_2026_Datawarehousing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [116]:
import pandas as pd
import numpy as np

In [117]:
!wget -O 'causes-of-road-traffic-crashes-in-uganda-2011-2020.xlsx' 'https://github.com/emmanuelyardamah/Data-Warehouse-and-Data-Mining/raw/main/causes-of-road-traffic-crashes-in-uganda-2011-2020.xlsx'

--2026-10-23 23:32:31--  https://github.com/emmanuelyardamah/Data-Warehouse-and-Data-Mining/raw/main/causes-of-road-traffic-crashes-in-uganda-2011-2020.xlsx
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/emmanuelyardamah/Data-Warehouse-and-Data-Mining/main/causes-of-road-traffic-crashes-in-uganda-2011-2020.xlsx [following]
--2026-10-23 23:32:32--  https://raw.githubusercontent.com/emmanuelyardamah/Data-Warehouse-and-Data-Mining/main/causes-of-road-traffic-crashes-in-uganda-2011-2020.xlsx
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13115 (13K) [application/octet-stream]
Saving to: ‘causes-of

In [118]:
file_path = 'causes-of-road-traffic-crashes-in-uganda-2011-2020.xlsx'
raw_df = pd.read_excel(file_path, header=None)

In [119]:
# Combine row 0 (Year) and row 1 (Severity) to form distinct column names
raw_df.iloc[0] = raw_df.iloc[0].ffill()
columns = []
for col_idx in range(len(raw_df.columns)):
    cat = str(raw_df.iloc[0, col_idx]).strip().replace('.0', '')
    subcat = str(raw_df.iloc[1, col_idx]).strip().replace(' ', '_')
    col_name = f"{cat}_{subcat}" if 'Causes' not in cat and cat != 'nan' else subcat
    columns.append(col_name)

df = raw_df.iloc[2:].copy()
df.columns = columns
df = df.rename(columns={df.columns[0]: 'Causes_of_Crashes'})


/tmp/ipykernel_8039/1512076343.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  raw_df.iloc[0] = raw_df.iloc[0].ffill()


In [120]:
# Ensure all numeric columns are floats/ints
for col in df.columns:
    if col != 'Causes_of_Crashes':
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)


In [121]:
# Find all columns containing annual totals
total_cols = [col for col in df.columns if 'Total' in col]

# Calculate a Grand Total across the years for each cause
df['Total_Accidents'] = df[total_cols].sum(axis=1)

# Categorize causes (rows) into the specific factors
def get_category(row_name):
    row_name = str(row_name).lower().strip()
    # Human behaviors
    if any(kw in row_name for kw in ['driving', 'pedestrian', 'speed', 'passenger', 'drink', 'drug', 'fatigue', 'influence']):
        return 'Population_Behavior'
    # Vehicle conditions
    elif any(kw in row_name for kw in ['dmc', 'vehicle', 'brake', 'tyre', 'overload', 'loading']):
        return 'Motor_Vehicle_Condition'
    # Road/Environment conditions
    else:
        return 'Road_Environment'

df['Category'] = df['Causes_of_Crashes'].apply(get_category)


In [122]:
# Pivot or aggregate so we have columns for each category across the years
years_data = []
years = sorted(list(set([col.split('_')[0] for col in total_cols if col.split('_')[0].isdigit()])))

for yr in years:
    yr_total_col = f"{yr}_Total"
    if yr_total_col in df.columns:
        behavior_sum = df[df['Category'] == 'Population_Behavior'][yr_total_col].sum()
        environment_sum = df[df['Category'] == 'Road_Environment'][yr_total_col].sum()
        vehicle_sum = df[df['Category'] == 'Motor_Vehicle_Condition'][yr_total_col].sum()
        total_sum = behavior_sum + environment_sum + vehicle_sum

        years_data.append({
            'Year': int(yr),
            'Population_Behavior': behavior_sum,
            'Road_Environment': environment_sum,
            'Motor_Vehicle_Condition': vehicle_sum,
            'Total_Road_Traffic_Accidents': total_sum
        })

cleaned_df = pd.DataFrame(years_data)


In [123]:
# Save the cleaned time-series data
cleaned_df.to_csv('cleaned_uganda_crashes_2011_2020.csv', index=False)
cleaned_df.to_excel('cleaned_uganda_crashes_2011_2020.xlsx', index=False)
print("Cleaned dataset successfully! Rows represent years, columns represent the 4 factors.")
print(cleaned_df.head())


Cleaned dataset successfully! Rows represent years, columns represent the 4 factors.
   Year  Population_Behavior  Road_Environment  Motor_Vehicle_Condition  \
0  2011              17981.0           25521.0                    761.0   
1  2012              15535.0           24368.0                    556.0   
2  2013              15494.0           22381.0                    593.0   
3  2014              16398.0           23589.0                    627.0   
4  2015              15923.0           23104.0                    665.0   

   Total_Road_Traffic_Accidents  
0                       44263.0  
1                       40459.0  
2                       38468.0  
3                       40614.0  
4                       39692.0  
